In [1]:
import openai
import pinecone
import json

from pinecone import Pinecone
from openai import OpenAI

In [2]:
import os

from dotenv import load_dotenv
load_dotenv()

pinecone_api_key = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key = pinecone_api_key)
print("Pinecone client successfully configured.")
print(pinecone_api_key[:5])

openai_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key = openai_api_key)
print("OpenAI client successfully configured.")
print(openai_api_key[:5])

index_name = "faq-database"
index = pc.Index(index_name)

Pinecone client successfully configured.
pcsk_
OpenAI client successfully configured.
sk-pr


In [3]:
def embedding_model(query, openai_client, model="text-embedding-3-small"):
  response = openai_client.embeddings.create(
      model=model,
      input=query
  )

  embedding = response.data[0].embedding
  return embedding

In [4]:
system_prompt = {
                    "role": "system",
                    "content": f"""
                    You are a helpfull E-Commerce assistant helping customers with their general questions regarding policies and procedures when buying in our store.
                    Our store sells e-books and courses for IT professionals.
                    """,
                }

def prompt_builder(system_message, context):
  return system_message["content"].format(context)

In [5]:
def retrieve_faq(query_embedding, index, top_k=1):
    response = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True,
        namespace="ns1"
    )
    return response['matches'][0]['metadata']['answer']

In [6]:
def generate_hypothetical_document(query, openai_client):
    prompt = f"Create a hypothetical document based on the following query: {query}"
    messages = [{"role": "system", "content": prompt}]

    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        max_tokens=300,
        temperature=0
    )

    return response.choices[0].message.content

In [7]:
def hypo_chatbot(query, openai_client, index):

    hypo_candidate = generate_hypothetical_document(query, openai_client)

    candidate_embedding = embedding_model(hypo_candidate, openai_client)

    best_match = retrieve_faq(candidate_embedding, index)

    augmented_prompt = prompt_builder(system_prompt, best_match)

    messages = [{"role": "system","content": augmented_prompt},
                {"role": "user","content": query}]

    response = openai_client.chat.completions.create(
      model="gpt-4o",
      messages=messages,
      max_tokens=250,
      temperature=0
    )

    return response.choices[0].message.content

In [8]:
response = hypo_chatbot("do you wrap some presents?", client, index)
print("Generated Response:", response)

Generated Response: Currently, we do not offer gift wrapping services for our e-books and courses. Since our products are digital, they are delivered electronically via email or through our platform, making traditional gift wrapping unnecessary. However, you can always personalize your gift by sending a thoughtful message along with the digital product. If you have any other questions or need assistance, feel free to ask!
